# 12장 기기 탑재와 날씨 신호등

이 노트북은 웹 교재 [12장 기기 탑재와 날씨 신호등](https://pidocs.aifusionlab.org/ai-python/ch12/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 실습 자료 내려받기

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/gesture_example.csv
!ls *.csv

## 12.3 교차 평가

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch12/12-3/)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score

df = pd.read_csv("gesture_example.csv")
W, STEP = 20, 10
FEATS = ["ax_mean", "ay_mean", "az_mean", "ax_std", "ay_std", "az_std", "mag_max"]

def window_features(w):
    mag = np.sqrt(w["ax"] ** 2 + w["ay"] ** 2 + w["az"] ** 2)
    return {"ax_mean": w["ax"].mean(), "ay_mean": w["ay"].mean(), "az_mean": w["az"].mean(),
            "ax_std": w["ax"].std(), "ay_std": w["ay"].std(), "az_std": w["az"].std(),
            "mag_max": mag.max(), "mag_mean": mag.mean(), "mag_std": mag.std()}

rows = []
for (student, gesture, rep), d in df.groupby(["student", "gesture", "rep"]):
    d = d.reset_index(drop=True)
    for start in range(0, len(d) - W + 1, STEP):
        f = window_features(d.iloc[start:start + W])
        f.update(student=student, gesture=gesture, rep=rep)
        rows.append(f)
F = pd.DataFrame(rows)

def loso(features):
    result = {}
    for s in sorted(F["student"].unique()):
        trn, tst = F[F["student"] != s], F[F["student"] == s]
        m = DecisionTreeClassifier(max_depth=3, random_state=0).fit(trn[features], trn["gesture"])
        result[s] = round(accuracy_score(tst["gesture"], m.predict(tst[features])), 3)
    return result

acc = loso(FEATS)
print(acc)
print("평균", round(np.mean(list(acc.values())), 3))

In [ ]:
trn, tst = F[F["student"] != "S04"], F[F["student"] == "S04"]
m = DecisionTreeClassifier(max_depth=3, random_state=0).fit(trn[FEATS], trn["gesture"])
print(export_text(m, feature_names=FEATS, decimals=2))
pd.crosstab(tst["gesture"], m.predict(tst[FEATS]), rownames=["actual"], colnames=["pred"])

In [ ]:
F.groupby(["gesture", "student"])[["az_mean", "ax_std", "ay_std"]].mean().round(2).unstack()

In [ ]:
F["std_sum"] = F["ax_std"] + F["ay_std"] + F["az_std"]
ROBUST = ["mag_mean", "mag_std", "mag_max", "std_sum"]
acc2 = loso(ROBUST)
print(acc2)
print("평균", round(np.mean(list(acc2.values())), 3))